In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [3]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [4]:
workflow_deep_dive = workflow_research('2026-03-04','2026-03-05')

In [5]:
workflow_deep_dive

[{'event_id': '1915_2026-03-01',
  'output': 'Rewarx announced RewarxStudio, a prompt-free AI product-photography platform that converts everyday snapshots into studio-quality 4K product images and cinematic video. Key features: prompt-free intent interface (reduces need for prompt engineering), snapshot-to-4K conversion, physically accurate rendering tuned for e-commerce materials, automated batch production and cinematic video generation. Targeted at online retailers and marketplace sellers to scale visual content production and lower studio costs; directs readers to rewarx.com for details.',
  'sources': [{'url': 'https://natlawreview.com/press-releases/rewarx-launches-prompt-free-ai-platform-automate-4k-e-commerce-product',
    'name': 'National Law Review'}],
  'news_date': '2026-03-01',
  'topic': 'Workflow improvement'},
 {'event_id': '1927_2026-03-02',
  'output': 'AMD announced Ryzen AI PRO 400 Series mobile workstation processors and new AI-PC options for consumers and busine

In [6]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [7]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [9]:
b = [ {'event_id': '1928_2026-03-02',
  'output': "Report on Google's newest AI agents (announced at MWC coverage) showing telcos adopting agent-driven automation for network operations — agents can detect drops in performance and trigger remediation actions (e.g., rerouting or restoring call quality), bringing telcos closer to autonomous network operations and promising measurable reductions in incident response time and service disruptions.",
  'sources': [{'url': 'https://siliconangle.com/2026/03/02/googles-newest-ai-agents-bring-telcos-step-closer-autonomous-network-operations/',
    'name': 'SiliconANGLE'}],
  'news_date': '2026-03-02',
  'topic': 'Workflow improvement'},
 {'event_id': '1944_2026-03-02',
  'output': 'TM Forum announced advancement of its AI‑Native Blueprint and launched the first core operational projects for AI at scale (press distributed March 2, 2026). The initiative targets trusted, production‑scale AI for telecom operators and service providers by defining operational practices, governance and tooling to move from pilots to large‑scale deployments — a material step for enterprises and telcos seeking measurable, governed AI operations that affect critical workflows and network/IT service availability.',
  'sources': [{'url': 'https://www.registerguard.com/press-release/story/38258/tm-forum-advances-ai-native-blueprint-with-launch-of-first-core-operational-projects-for-ai-at-scale/',
    'name': 'Register Guard'}],
  'news_date': '2026-03-02',
  'topic': 'Workflow improvement'},
 {'event_id': '1984_2026-03-02',
  'output': 'StorageChain launched a BYOC (Bring Your Own Cloud) infrastructure layer for enterprise AI (press release Mar 2, 2026), enabling enterprises to run AI intelligence layers over existing storage across multiple clouds without forced data consolidation — important for governance, compliance, cost control and enabling measurable analytics on long-lived operational datasets.',
  'sources': [{'url': 'https://www.providencejournal.com/press-release/story/36279/storagechain-launches-byoc-infrastructure-layer-for-enterprise-ai/',
    'name': 'Providence Journal'}],
  'news_date': '2026-03-02',
  'topic': 'Workflow improvement'},
 {'event_id': '1988_2026-03-02',
  'output': 'TechCrunch reports Stripe released a preview tool to help companies track, pass through, and potentially profit from underlying AI model fees — a new billing/monetization feature aimed at reducing friction for AI‑powered businesses.',
  'sources': [{'url': 'https://techcrunch.com/2026/03/02/stripe-wants-to-turn-your-ai-costs-into-a-profit-center/',
    'name': 'TechCrunch'}],
  'news_date': '2026-03-02',
  'topic': 'Workflow improvement'},
 {'event_id': '1993_2026-03-02',
  'output': 'EY announced EY.ai Agentic for Sales, an agentic sales-orchestration platform launched in collaboration with Snowflake and Canva. The release positions the product to address enterprise AI fragmentation by orchestrating agent workflows across data (Snowflake) and creative assets (Canva), intended to streamline sales processes and enable integrated agent-driven automations for enterprise sales teams. The announcement does not include pricing or specific GA/access dates in the press release.',
  'sources': [{'url': 'https://www.ey.com/en_gl/newsroom/2026/03/ey-in-collaboration-with-snowflake-and-canva-announces-launch-of-agentic-sales-orchestration-platform-to-address-enterprise-ai-fragmentation',
    'name': 'EY'}],
  'news_date': '2026-03-02',
  'topic': 'Workflow improvement'},
 {'event_id': '2062_2026-03-03',
  'output': 'Dialpad announced product advancements to its Agentic AI platform (press release) that help enterprises identify high-impact AI use cases, validate ROI before deployment (Proving Ground), and measure operational outcomes (closed-loop analytics tied to resolution, average handle time, CSAT). Dialpad cites customer examples (HOPCo) reporting reduced resolution times and improved patient satisfaction and lists usage metrics (e.g., 775M AI recaps, 450M AI CSAT scores) as evidence of measurable impact and scale. The release is dated March 3, 2026 and frames the capability as explicitly intended to move pilots into production with governance and measurable outcomes.',
  'sources': [{'url': 'https://www.dialpad.com/press/agentic-ai-platform-to-empower-enterprises/',
    'name': 'Dialpad'}],
  'news_date': '2026-03-03',
  'topic': 'Workflow improvement'},
 {'event_id': '2076_2026-03-03',
  'output': 'OpenAI (ChatGPT) release notes — GPT-5.3 Instant update (Mar 3, 2026): OpenAI published ChatGPT release notes describing GPT-5.3 Instant, a quality-of-answer update that delivers more accurate answers, richer and better-contextualized web-search results, and fewer unnecessary caveats or overly declarative phrasing. The update emphasizes improved tone, relevance and conversational flow — areas directly affecting how professionals use ChatGPT for research, content/brief generation, design critiques, product specs and automation. The release notes also include other recent product updates (projects, deep research, interactive code blocks) but do not announce pricing or new access tiers in this entry.',
  'sources': [{'url': 'https://help.openai.com/en/articles/6825453-chatgpt-release-notes',
    'name': 'OpenAI'}],
  'news_date': '2026-03-03',
  'topic': 'Workflow improvement'}]

In [10]:
openai_research_workflow(b)

- REPORT SNAPSHOT: Nokia published a press release titled “Nokia expands Network as Code ecosystem, advances API-based agentic AI with Google Cloud #MWC26” on March 3, 2026 (Espoo, Finland, 09:00 AM CEST), detailing the expansion of Network as Code (NaC) and the integration of Google Cloud’s agentic AI with Nokia’s platform. Scope includes global telcos, a partner ecosystem of 75+ members, and the introduction of MCP/ADK-based agent workflows using Google Gemini models; notes exposure of Nokia APIs on Google Cloud Marketplace (launched July 14, 2025). [Source: Nokia, https://www.nokia.com/newsroom/nokia-expands-network-as-code-ecosystem-advances-api-based-agentic-ai-with-google-cloud-mwc26/], [Source: Nokia, https://www.nokia.com/about-us/news/releases/2025/07/14/nokia-network-apis-now-available-on-google-cloud-marketplace-making-it-even-easier-for-developers-to-utilize/]([nokia.com](https://www.nokia.com/newsroom/nokia-expands-network-as-code-ecosystem-advances-api-based-agentic-ai-wi